# Thiết kế lại bộ chart cho COO: từ "slide xấu" tới một trang storytelling

Bộ `bike-sharing-uci`, `day.csv`: 731 ngày (01/01/2011 tới 31/12/2012) của Capital Bikeshare, Washington DC.
`cnt` là tổng lượt thuê, tách `casual` (vãng lai) và `registered` (thành viên); `weathersit` 1 = quang,
2 = mây/sương, 3 = mưa hoặc tuyết nhẹ, 4 = mưa to.

Đồng nghiệp gửi COO một slide "tổng kết 2 năm" (cell `explore` bên dưới vẽ lại): một pie theo mùa, một chart
dual axis nhiệt độ và lượt thuê, và một bảng số dày đặc. COO trả lời: **"Tôi không biết phải nhìn vào đâu.
Năm nay mình làm tốt hơn không, và cái gì làm mình mất khách?"**

Việc của bạn: thiết kế lại thành **một trang** trả lời đúng hai câu đó, rồi viết `MEMO.md` giải thích bạn
giữ, đổi, bỏ gì và vì sao.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật, vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/08-visualization-chart-reading/exercises/advanced/02-redesign-for-stakeholder -q` (kiểm hai hàm số liệu, cấu trúc trang, và `MEMO.md` đủ mục). Chất lượng
  thiết kế chấm theo rubric trong lời giải.

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc day.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
day = pd.read_csv(BIKE / "day.csv")
print(day.shape, day.dteday.min(), "→", day.dteday.max())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Slide gốc gửi COO

Chạy cell dưới để thấy slide gốc. Trước khi sửa, ghi ra (trong `MEMO.md`, mục "Chart cũ") mỗi chart sai ở
đâu theo các nguyên tắc của module: pie nhiều lát gần bằng nhau, dual axis với hai thang tuỳ ý, bảng số
thay cho chart, tiêu đề mô tả thay vì kết luận, màu không mang nghĩa, nhãn mùa của UCI lệch một mùa.

In [ ]:
# explore: slide gốc - đừng sửa cell này, nó là "trước"
SEASONS = {1: "Spring", 2: "Summer", 3: "Fall", 4: "Winter"}          # nhãn gốc của UCI (lệch một mùa, xem Basic 02)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
by_season = day.groupby("season")["cnt"].sum()
axes[0].pie(by_season, labels=[SEASONS[s] for s in by_season.index], autopct="%1.1f%%", startangle=90,
            colors=["#ff9999", "#66b3ff", "#99ff99", "#ffcc99"])
axes[0].set_title("Rentals by season")

monthly = day.assign(m=pd.to_datetime(day.dteday).dt.to_period("M").dt.to_timestamp()).groupby("m")[["cnt", "temp"]].agg({"cnt": "sum", "temp": "mean"})
axes[1].bar(monthly.index, monthly.cnt, width=20, color="orange")
twin = axes[1].twinx()
twin.plot(monthly.index, monthly.temp * 41, color="red", marker="s")
twin.set_ylim(5, 32)
axes[1].set_ylim(30_000, 230_000)
axes[1].set_title("Rentals and Temperature")

table = day.assign(year=day.yr.map({0: 2011, 1: 2012})).groupby(["year", "mnth"])[["casual", "registered", "cnt", "temp", "hum", "windspeed"]].mean().round(2)
axes[2].axis("off")
axes[2].table(cellText=table.head(12).values, colLabels=table.columns, loc="center", fontsize=6)
axes[2].set_title("Data")
fig.suptitle("Bike Sharing Report 2011-2012")
plt.show()

## 2. Số liệu cho trang mới

`yoy_by_month(day) -> pd.DataFrame`: tổng `cnt` theo tháng lịch của hai năm có trong `day` (dựa trên
`dteday`, không dùng cột `yr`). Index là tháng `1..12` (tên `month`); cột theo thứ tự `prev` (năm đầu),
`curr` (năm sau), `growth = curr / prev - 1`.

In [ ]:
def yoy_by_month(day: pd.DataFrame) -> pd.DataFrame:
    """Lượt thuê theo tháng của hai năm: index month 1..12, cột prev, curr, growth."""
    # TODO: năm và tháng từ dteday; tổng cnt; pivot năm thành cột; growth = curr / prev - 1
    raise NotImplementedError

`weather_effect(day) -> pd.Series`: lượt thuê ngày thời tiết xấu thấp hơn ngày đẹp bao nhiêu, **sau khi
trừ mùa vụ và tăng trưởng**. So thẳng TB theo `weathersit` là so khập khiễng: ngày mưa tập trung vào một số
tháng, và năm sau đông khách hơn năm trước. Cách làm:

1. mức nền của mỗi tháng (năm-tháng lịch) = TB `cnt` của các ngày `weathersit == 1` trong tháng đó;
2. tỉ số của từng ngày = `cnt` / mức nền tháng của ngày đó (tháng không có ngày quang thì bỏ ngày đó);
3. kết quả = TB tỉ số theo `weathersit`, trừ 1. Index là `weathersit` (tăng dần), tên Series `effect`.

`weathersit == 1` vì thế luôn quanh 0; `-0.5` nghĩa là "mất một nửa lượt so với ngày quang cùng tháng".

In [ ]:
def weather_effect(day: pd.DataFrame) -> pd.Series:
    """Lượt thuê so với ngày trời quang cùng tháng: index weathersit, tên "effect"."""
    # TODO: mức nền = TB cnt của ngày weathersit 1 trong cùng năm-tháng; tỉ số từng ngày; TB theo weathersit; trừ 1
    raise NotImplementedError

## 3. Trang mới

`build_page(day) -> matplotlib.figure.Figure`: vẽ trang storytelling cho COO và trả `Figure`. Test kiểm
phần "luật chơi" có thể đo được; chất lượng thiết kế chấm theo rubric:

- `fig.suptitle` là **headline có con số** (kết luận, không phải tên báo cáo);
- 2-4 chart (axes đang hiện), mỗi chart có tiêu đề;
- không pie, không dual axis (`twinx`), không bảng (`ax.table`);
- ít nhất một annotation (`ax.annotate`) giải thích điểm cần chú ý ngay trên chart.

Số trong tiêu đề phải tính từ `day` (hàm được chấm trên dữ liệu khác). Gợi ý thứ tự: chart kết luận (năm nay
so với năm trước) trước, chart "vì sao / rủi ro" sau. Câu hỏi về tỉ trọng mùa hay thành viên: có thật sự cần
trên trang này không?

In [ ]:
def build_page(day: pd.DataFrame) -> plt.Figure:
    """Một trang storytelling cho COO: suptitle là headline có con số, 2-4 chart, không pie, không dual axis."""
    # TODO: tính số bằng yoy_by_month / weather_effect; vẽ; trả fig (đừng plt.show() trong hàm)
    raise NotImplementedError

In [ ]:
# explore: xem trang mới và các con số
print(yoy_by_month(day))
print(weather_effect(day).round(3))
fig = build_page(day)
plt.show()

## 4. Viết `MEMO.md`

Mở `starter/MEMO.md`, điền theo khung: headline của trang, người đọc và hai câu hỏi, thứ tự các chart trên
trang mới, chart cũ giữ / đổi / bỏ và vì sao, kiểm tra năm giây, giới hạn. 150-500 từ, xoá mọi dòng `TODO`.